In [23]:
%pip install databricks-langchain databricks-vectorsearch

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/40.1 kB ? eta -:--:--
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/90.6 kB ? eta -:--:--
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 7.8 MB/s eta 0:00:00
  Using cached requests-2.32.5-py3-none-any.whl.metadata (4.9 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/764.2 kB ? eta -:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 764.2/764.2 kB 33.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/102.2 kB ? eta -:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.2/102.2 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/8.9 MB ? eta -:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸━━━━ 7.9/8.9 MB 181.1 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 8.9/8.9 MB 167.5 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 8.9/8.9 MB 167.5 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━

Error: The "path" argument must be of type string. Received type undefined

In [24]:
%restart_python

In [26]:
"""
Unity Catalog Functions for Multi-Agent System

This module contains UC function implementations for the custom agents:
- Query Planning and Analysis
- SQL Synthesis
- SQL Execution

These functions can be registered in Unity Catalog and called by the LangGraph supervisor.
"""

import json
from typing import Dict, List, Optional, Any
# from databricks.sdk.runtime import spark

In [27]:

# 1. Function to query delta table by field
def query_delta_table(table_name: str, filter_field: str, filter_value: str, select_fields: List[str] = None) -> Any:
    """
    Query a delta table with a filter condition.
    
    Args:
        table_name: Full table name (catalog.schema.table)
        filter_field: Field name to filter on
        filter_value: Value to filter by
        select_fields: List of fields to select (None = all fields)
    
    Returns:
        Spark DataFrame with query results
    """
    if select_fields:
        fields_str = ", ".join(select_fields)
    else:
        fields_str = "*"
    
    df = spark.sql(f"""
        SELECT {fields_str}
        FROM {table_name}
        WHERE {filter_field} = '{filter_value}'
    """)
    
    return df

# 2. Call the function to get space_summary chunks
table_name = "yyang.multi_agent_genie.enriched_genie_docs_chunks"
space_summary_df = query_delta_table(
    table_name=table_name,
    filter_field="chunk_type",
    filter_value="space_summary",
    select_fields=["space_id", "space_title", "searchable_content"]
)

# Display the table
print("Space Summary Data from Delta Table:")
print("="*80)
space_summary_df.show(truncate=False)
print("="*80)

# 3. Convert table to JSON with one entry per space
space_summary_list = space_summary_df.collect()
context = {}

for row in space_summary_list:
    space_id = row["space_id"]
    context[space_id] = row["searchable_content"]
    

# Display the context JSON
print("\nContext JSON (per space):")
print("="*80)
print(json.dumps(context, indent=2))
print("="*80)
print()

Space Summary Data from Delta Table:
+--------------------------------+------------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [28]:
context

{'01f0956a387714969edde65458dcc22a': 'Space: HealthVerityClaims\nSpace ID: 01f0956a387714969edde65458dcc22a\n\nDescription: This Genie space provides comprehensive healthcare claims analytics by combining medical and pharmacy claims data across different insurance payer types (Commercial, Medicare, and Medicaid). It enables analysis of healthcare utilization patterns, treatment costs, medication adherence, and care delivery across various settings including hospitals, emergency rooms, and pharmacies. Users can query patient care episodes, track prescription patterns and refills, analyze healthcare spending, compare costs across payer types, and gain insights into both medical service utilization and pharmaceutical dispensing trends.\n\nAvailable Tables (2 total):\n• healthverity_claims_sample_patient_dataset.hv_claims_sample.medical_claim (6 columns)\n  Description: This table contains medical insurance claims data tracking healthcare services provided to patients across various care s

In [29]:

from databricks_langchain import ChatDatabricks

# Initialize LLM for analysis
llm = ChatDatabricks(endpoint="databricks-claude-sonnet-4-5")


2025-12-13 22:51:52.259092: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-12-13 22:51:52.270132: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-12-13 22:51:52.297004: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-12-13 22:51:52.311944: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-12-13 22:51:52.331202: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been 

upon restart run until here

In [ ]:

query = "Provide me three questions each need to be answered by joining two or more Genie spaces."

# Step 1: Check query clarity
json_template = {
    "questions": [
        {
            "question": "What is the average cost of medical claims for patients diagnosed with diabetes, broken down by insurance payer type and patient age group?",
            "spaces_required": [
                "HealthVerityClaims",
                "HealthVerityProcedureDiagnosis",
                "HealthVerityProviderEnrollment"
            ],
            "reasoning": "Requires joining medical_claim (HealthVerityClaims) for cost data and payer type, diagnosis (HealthVerityProcedureDiagnosis) for diabetes ICD-10 codes, and enrollment (HealthVerityProviderEnrollment) for patient birth year to calculate age groups"
        },
        {
            "question": "Which medical procedures have the highest pharmacy claim costs within 30 days post-procedure, and what are the most commonly prescribed medications?",
            "spaces_required": [
                "HealthVerityClaims",
                "HealthVerityProcedureDiagnosis"
            ],
            "reasoning": "Requires joining procedure (HealthVerityProcedureDiagnosis) for CPT/HCPCS procedure codes and service dates with pharmacy_claim (HealthVerityClaims) for medication NDC codes and costs, matching by patient_id and date ranges"
        }
    ]
}

clarity_prompt = f"""
You are a helpful assistant that can analyze a user query and provide a JSON response with the following structure:
{json.dumps(json_template, indent=2)}

Question: {query}

Context: {json.dumps(context, indent=2)}

Provide your answer in JSON format with the same structure as above.

Only return valid JSON, no explanations.
"""

clarity_response = llm.invoke(clarity_prompt)

json_str = clarity_response.content.strip('```json')
# Parse JSON
try:
    clarity_result = json.loads(json_str)
    print("Parsed clarity result:")
    print(json.dumps(clarity_result, indent=2))
except json.JSONDecodeError as e:
    print(f"JSON parsing error: {e}")
    print(f"Attempted to parse: {json_str[:200]}...")
    raise

In [50]:

"""
Analyze a user query and create an execution plan.

This function:
1. Checks if the query is clear or needs clarification
2. Searches for relevant Genie spaces using vector search
3. Determines execution strategy (single/multi-space, join requirements)

Args:
    query: The user's question
    vector_search_index: Full name of the vector search index
    num_results: Number of relevant spaces to retrieve
    
Returns:
    JSON string with QueryPlan structure containing:
    - question_clear: bool
    - clarification_needed: str (if applicable)
    - clarification_options: list[str] (if applicable)
    - sub_questions: list[str]
    - requires_multiple_spaces: bool
    - relevant_space_ids: list[str]
    - requires_join: bool
    - join_strategy: str ("fast_route" or "slow_route")
    - execution_plan: str
"""
from databricks_langchain import ChatDatabricks

# Initialize LLM for analysis
llm = ChatDatabricks(endpoint="databricks-claude-sonnet-4-5")

query = "How many patients are ?"
query = "How many patients are in the dataset (total unique patients)?"

# Step 1: Check query clarity
# TODO: include {context} in the prompt, context could be some part of the VS results that is relevant to the question
clarity_prompt = f"""
Analyze the following question for clarity and specificity based on the context.

Question: {query}

Context: {json.dumps(context, indent=2)}


Determine if:
1. The question is clear and answerable as-is
2. The question needs clarification

If clarification is needed, provide:
- A brief explanation of what's unclear
- 2-3 specific clarification options the user can choose from

Return your analysis as JSON:
{{
    "question_clear": true/false,
    "clarification_needed": "explanation if unclear",
    "clarification_options": ["option 1", "option 2", "option 3"]
}}

Only return valid JSON, no explanations.
"""

clarity_response = llm.invoke(clarity_prompt)

Trace(trace_id=tr-fc46eed3e4bc859785d7402e5c6e79a7)

In [51]:
print(clarity_prompt)

Analyze the following question for clarity and specificity based on the context.

Question: How many patients are in the dataset (total unique patients)?

Context: {
  "01f0956a387714969edde65458dcc22a": "Space: HealthVerityClaims\nSpace ID: 01f0956a387714969edde65458dcc22a\n\nDescription: This Genie space provides comprehensive healthcare claims analytics by combining medical and pharmacy claims data across different insurance payer types (Commercial, Medicare, and Medicaid). It enables analysis of healthcare utilization patterns, treatment costs, medication adherence, and care delivery across various settings including hospitals, emergency rooms, and pharmacies. Users can query patient care episodes, track prescription patterns and refills, analyze healthcare spending, compare costs across payer types, and gain insights into both medical service utilization and pharmaceutical dispensing trends.\n\nAvailable Tables (2 total):\n\u2022 healthverity_claims_sample_patient_dataset.hv_claim

In [52]:
clarity_response

AIMessage(content='```json\n{\n    "question_clear": false,\n    "clarification_needed": "The patient_id field appears across multiple tables in different spaces. The question could be asking for unique patients across all tables, or unique patients within a specific space or table context.",\n    "clarification_options": [\n        "Count unique patients across all tables in all spaces (medical_claim, pharmacy_claim, diagnosis, procedure, enrollment, provider)",\n        "Count unique patients in the HealthVerityClaims space only (medical_claim and pharmacy_claim tables)",\n        "Count unique patients in the enrollment table (which likely represents the master patient list)"\n    ]\n}\n```', additional_kwargs={}, response_metadata={'usage': {'prompt_tokens': 1780, 'completion_tokens': 152, 'total_tokens': 1932}, 'prompt_tokens': 1780, 'completion_tokens': 152, 'total_tokens': 1932, 'model': 'us.anthropic.claude-sonnet-4-5-20250929-v1:0', 'model_name': 'us.anthropic.claude-sonnet-4-

In [53]:

# Extract JSON from the response (handle markdown code blocks)
response_text = clarity_response.content

# Debug: print raw response
print("Raw LLM Response:")
print(response_text)
print("\n" + "="*80 + "\n")

Raw LLM Response:
```json
{
    "question_clear": false,
    "clarification_needed": "The patient_id field appears across multiple tables in different spaces. The question could be asking for unique patients across all tables, or unique patients within a specific space or table context.",
    "clarification_options": [
        "Count unique patients across all tables in all spaces (medical_claim, pharmacy_claim, diagnosis, procedure, enrollment, provider)",
        "Count unique patients in the HealthVerityClaims space only (medical_claim and pharmacy_claim tables)",
        "Count unique patients in the enrollment table (which likely represents the master patient list)"
    ]
}
```


In [54]:

# # Try to extract JSON from markdown code blocks
# import re
# json_match = re.search(r'```(?:json)?\s*\n?(.*?)\n?```', response_text, re.DOTALL)
# if json_match:
#     json_str = json_match.group(1).strip()
# else:
#     # If no code blocks, try to use the whole response
#     json_str = response_text.strip()

json_str = response_text.strip('```json')
# Parse JSON
try:
    clarity_result = json.loads(json_str)
    print("Parsed clarity result:")
    print(json.dumps(clarity_result, indent=2))
except json.JSONDecodeError as e:
    print(f"JSON parsing error: {e}")
    print(f"Attempted to parse: {json_str[:200]}...")
    raise

Parsed clarity result:
{
  "question_clear": false,
  "clarification_needed": "The patient_id field appears across multiple tables in different spaces. The question could be asking for unique patients across all tables, or unique patients within a specific space or table context.",
  "clarification_options": [
    "Count unique patients across all tables in all spaces (medical_claim, pharmacy_claim, diagnosis, procedure, enrollment, provider)",
    "Count unique patients in the HealthVerityClaims space only (medical_claim and pharmacy_claim tables)",
    "Count unique patients in the enrollment table (which likely represents the master patient list)"
  ]
}

In [35]:
vector_search_index: str = "yyang.multi_agent_genie.enriched_genie_docs_chunks_vs_index"
num_results: int = 5
query = "What is the average cost of medical claims for patients diagnosed with diabetes, broken down by insurance payer type and patient age group?"
query = "How many patients are 65 years old and above?"
query = "How many medical claims are above $500? How many Rx claims are above $1000?"
query = "What is the average cost of medical claims for patients diagnosed with diabetes, broken down by insurance payer type and patient age group?"

# Step 2: Search for relevant Genie spaces using AI Bridge VectorSearchRetrieverTool
from databricks_langchain import VectorSearchRetrieverTool

# Create VectorSearchRetrieverTool with filter for space_summary chunks
vs_tool = VectorSearchRetrieverTool(
    index_name=vector_search_index,
    num_results=num_results,
    columns=["space_id", "space_title", "searchable_content"],
    filters={"chunk_type": "space_summary"},
    query_type="ANN",
    include_metadata=True,
    include_score=True
)

# Invoke the tool to get results
docs = vs_tool.invoke({"query": query})

# Extract space information from document metadata
relevant_spaces = []
for doc in docs:
    relevant_spaces.append({
        "space_id": doc.metadata.get("space_id", ""),
        "space_title": doc.metadata.get("space_title", ""),
        "searchable_content": doc.page_content,
        "score": doc.metadata.get("score", 0.0)
    })

if not relevant_spaces:
    relevant_spaces = []

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.

In [36]:
relevant_spaces

[{'space_id': '01f0956a387714969edde65458dcc22a',
  'space_title': 'HealthVerityClaims',
  'searchable_content': 'Space: HealthVerityClaims\nSpace ID: 01f0956a387714969edde65458dcc22a\n\nDescription: This Genie space provides comprehensive healthcare claims analytics by combining medical and pharmacy claims data across different insurance payer types (Commercial, Medicare, and Medicaid). It enables analysis of healthcare utilization patterns, treatment costs, medication adherence, and care delivery across various settings including hospitals, emergency rooms, and pharmacies. Users can query patient care episodes, track prescription patterns and refills, analyze healthcare spending, compare costs across payer types, and gain insights into both medical service utilization and pharmaceutical dispensing trends.\n\nAvailable Tables (2 total):\n• healthverity_claims_sample_patient_dataset.hv_claims_sample.medical_claim (6 columns)\n  Description: This table contains medical insurance claims 

In [37]:
context

{'01f0956a387714969edde65458dcc22a': 'Space: HealthVerityClaims\nSpace ID: 01f0956a387714969edde65458dcc22a\n\nDescription: This Genie space provides comprehensive healthcare claims analytics by combining medical and pharmacy claims data across different insurance payer types (Commercial, Medicare, and Medicaid). It enables analysis of healthcare utilization patterns, treatment costs, medication adherence, and care delivery across various settings including hospitals, emergency rooms, and pharmacies. Users can query patient care episodes, track prescription patterns and refills, analyze healthcare spending, compare costs across payer types, and gain insights into both medical service utilization and pharmaceutical dispensing trends.\n\nAvailable Tables (2 total):\n• healthverity_claims_sample_patient_dataset.hv_claims_sample.medical_claim (6 columns)\n  Description: This table contains medical insurance claims data tracking healthcare services provided to patients across various care s

In [38]:
doc

Document(metadata={'space_id': '01f0956a4b0512e2a8aa325ffbac821b', 'space_title': 'HealthVerityProcedureDiagnosis', 'chunk_id': 21.0, 'score': 0.0025167104}, page_content='Space: HealthVerityProcedureDiagnosis\nSpace ID: 01f0956a4b0512e2a8aa325ffbac821b\n\nDescription: This Genie space provides healthcare claims data focused on medical diagnoses and procedures performed during patient encounters. It enables analysis of treatment patterns, healthcare utilization, and cost analytics by linking specific medical conditions (ICD-10 codes) with the procedures and services (CPT/HCPCS codes) delivered to patients. Users can query this space to understand disease prevalence, procedure frequencies, reimbursement patterns, and the relationship between diagnoses and treatments across healthcare encounters.\n\nAvailable Tables (2 total):\n• healthverity_claims_sample_patient_dataset.hv_claims_sample.diagnosis (7 columns)\n  Description: This table contains diagnosis information associated with heal

In [39]:
query = "What is the average cost of medical claims for patients diagnosed with diabetes, broken down by insurance payer type and patient age group?"

In [40]:
# Step 3: Create execution plan with relevant spaces
# at least, relevant spaces have sorted by score in descending order, this will be additional context for the LLM.
planning_prompt = f"""
You are a query planning expert. Analyze the following question and create an execution plan.

Question: {query}

Potentially relevant Genie spaces:
{json.dumps(relevant_spaces, indent=2)}

Break down the question and determine:
1. What are the sub-questions or analytical components?
2. How many Genie spaces are needed to answer completely? (List their space_ids)
3. If multiple spaces are needed, do we need to JOIN data across them?
4. If JOIN is needed, what's the best strategy:
    - "fast_route": Directly synthesize SQL across multiple tables
    - "slow_route": Query each space separately, then combine results
5. If no JOIN needed, can answers be verbally merged?

Return your analysis as JSON:
{{
    "question_clear": true,
    "sub_questions": ["sub-question 1", "sub-question 2", ...],
    "requires_multiple_spaces": true/false,
    "relevant_space_ids": ["space_id_1", "space_id_2", ...],
    "requires_join": true/false,
    "join_strategy": "fast_route" or "slow_route" or null,
    "execution_plan": "Brief description of execution plan"
}}

Only return valid JSON, no explanations.
"""

planning_response = llm.invoke(planning_prompt)
plan_result = json.loads(planning_response.content.strip('```json'))


Trace(trace_id=tr-223f5a7295db3f541f19ddf76e431197)

In [41]:
print(planning_prompt)

You are a query planning expert. Analyze the following question and create an execution plan.

Question: What is the average cost of medical claims for patients diagnosed with diabetes, broken down by insurance payer type and patient age group?

Potentially relevant Genie spaces:
[
  {
    "space_id": "01f0956a387714969edde65458dcc22a",
    "space_title": "HealthVerityClaims",
    "searchable_content": "Space: HealthVerityClaims\nSpace ID: 01f0956a387714969edde65458dcc22a\n\nDescription: This Genie space provides comprehensive healthcare claims analytics by combining medical and pharmacy claims data across different insurance payer types (Commercial, Medicare, and Medicaid). It enables analysis of healthcare utilization patterns, treatment costs, medication adherence, and care delivery across various settings including hospitals, emergency rooms, and pharmacies. Users can query patient care episodes, track prescription patterns and refills, analyze healthcare spending, compare costs ac

In [42]:
plan_result

{'question_clear': True,
 'sub_questions': ['Identify patients diagnosed with diabetes from diagnosis codes',
  'Calculate the cost of medical claims for diabetes patients',
  'Determine the insurance payer type for each claim',
  'Calculate patient age from birth year',
  'Group patients into age groups',
  'Calculate average claim cost by payer type and age group'],
 'requires_multiple_spaces': True,
 'relevant_space_ids': ['01f0956a4b0512e2a8aa325ffbac821b',
  '01f0956a387714969edde65458dcc22a',
  '01f0956a54af123e9cd23907e8167df9'],
 'requires_join': True,
 'join_strategy': 'fast_route',
 'execution_plan': 'Use fast_route to join across three spaces: (1) HealthVerityProcedureDiagnosis to identify diabetes patients via ICD-10 diagnosis codes and get claim costs from procedure table, (2) HealthVerityClaims to get payer type from medical_claim table, and (3) HealthVerityProviderEnrollment to get patient birth year from enrollment table to calculate age groups. Join on patient_id and c

In [8]:
# Step 3: Create execution plan with all spaces context.
planning_prompt = f"""
You are a query planning expert. Analyze the following question and create an execution plan.

Question: {query}

All Genie spaces:
{json.dumps(context, indent=2)}

Break down the question and determine:
1. What are the sub-questions or analytical components?
2. How many Genie spaces are needed to answer completely? (List their space_ids)
3. If multiple spaces are needed, do we need to JOIN data across them?
4. If JOIN is needed, what's the best strategy:
    - "fast_route": Directly synthesize SQL across multiple tables
    - "slow_route": Query each space separately, then combine results
5. If no JOIN needed, can answers be verbally merged?

Return your analysis as JSON:
{{
    "question_clear": true,
    "sub_questions": ["sub-question 1", "sub-question 2", ...],
    "requires_multiple_spaces": true/false,
    "relevant_space_ids": ["space_id_1", "space_id_2", ...],
    "requires_join": true/false,
    "join_strategy": "fast_route" or "slow_route" or null,
    "execution_plan": "Brief description of execution plan"
}}

Only return valid JSON, no explanations.
"""

planning_response = llm.invoke(planning_prompt)
plan_result = json.loads(planning_response.content.strip('```json'))


Trace(trace_id=tr-856a729631434376e823b44f665656dd)

In [9]:
plan_result

{'question_clear': True,
 'sub_questions': ['Identify patients diagnosed with diabetes using diagnosis codes',
  'Calculate the average cost of medical claims for these diabetes patients',
  'Break down the average cost by insurance payer type',
  'Break down the average cost by patient age group',
  'Determine patient age from enrollment data (birth year)'],
 'requires_multiple_spaces': True,
 'relevant_space_ids': ['01f0956a387714969edde65458dcc22a',
  '01f0956a4b0512e2a8aa325ffbac821b',
  '01f0956a54af123e9cd23907e8167df9'],
 'requires_join': True,
 'join_strategy': 'fast_route',
 'execution_plan': 'Join diagnosis table to identify diabetes patients (ICD-10 codes starting with E10-E14), join with medical_claim table to get claim costs and payer types, join with enrollment table to calculate patient age from birth year. Group by payer type and age group to calculate average costs.'}

In [ ]:
# from operator import itemgetter

# keys = ["relevant_space_ids", "execution_plan"]
# getter = itemgetter(*keys)
# execuation_plan = dict(zip(keys, getter(plan_result)))
# print(execuation_plan)  # {'a': 1, 'c': 3}

{'relevant_space_ids': ['01f0956a387714969edde65458dcc22a', '01f0956a4b0512e2a8aa325ffbac821b', '01f0956a54af123e9cd23907e8167df9'], 'execution_plan': 'Join diagnosis table to identify diabetes patients (ICD-10 codes starting with E10-E14), join with medical_claim table to get claim costs and payer types, join with enrollment table to calculate patient age from birth year. Group by payer type and age group to calculate average costs.'}

In [ ]:
# this is no-tool agent version
def synthesize_sql_fast_route(
    query: str,
) -> str:
    """
    Synthesize SQL query directly across multiple tables (fast route).
    
    Args:
        query: The user's question
        table_metadata_json: JSON string with table metadata including:
            - table_name
            - columns
            - relationships
            - sample_data
            
    Returns:
        SQL query string
    """

    prompt = f"""
    You are an expert SQL developer. Generate a SQL query to answer the following question
    using the available tables.
    
    Question: {query}

    Execution Plan:
    {json.dumps(plan_result, indent=2)}

    Available Tables and Columns Metadata from the Genie spaces summary level: 
    {json.dumps(relevant_spaces, indent=2)}

    Try if you can synthesize the SQL query directly from the available metadata already provided.

    If you are not sure, feel free to call tools you have access to get more drill-down information about the relevant tables and columns you can use in forming the SQL query.
    1. call sql UC function to query relevant table level metadata by filtering chunk_type == "table_overview"
    2. call sql UC function to query relevant column level metadata by filtering chunk_type == "column_detail"
    3. last resort, if still not enough, you can query all Metadata information lumped together from calling sql UC function to query chunk_type == "space_details". Be careful not to use it too often cause it will include a lot of tokens.
    
    Generate a complete, executable SQL query. Include:
    - Proper JOINs where needed
    - WHERE clauses for filtering
    - Appropriate aggregations
    - Column aliases for clarity
    
    Return ONLY the SQL query, no explanations or markdown formatting.
    """
    
    response = llm.invoke(prompt)
    print(prompt)
    sql = response.content.strip()
    
    # Remove markdown code blocks if present
    if sql.startswith("```"):
        lines = sql.split("\n")
        sql = "\n".join(lines[1:-1]) if len(lines) > 2 else sql
    
    return sql

In [51]:
sql_result = synthesize_sql_fast_route(query)

You are an expert SQL developer. Generate a SQL query to answer the following question
    using the available tables.
    
    Question: What is the average cost of medical claims for patients diagnosed with diabetes, broken down by insurance payer type and patient age group?

    Execution Plan:
    {
  "question_clear": true,
  "sub_questions": [
    "Identify patients diagnosed with diabetes from diagnosis codes",
    "Calculate the cost of medical claims for diabetes patients",
    "Determine the insurance payer type for each claim",
    "Calculate patient age from birth year",
    "Group patients into age groups",
    "Calculate average claim cost by payer type and age group"
  ],
  "requires_multiple_spaces": true,
  "relevant_space_ids": [
    "01f0956a4b0512e2a8aa325ffbac821b",
    "01f0956a387714969edde65458dcc22a",
    "01f0956a54af123e9cd23907e8167df9"
  ],
  "requires_join": true,
  "join_strategy": "fast_route",
  "execution_plan": "Use fast_route to join across three spa

In [52]:
print(sql_result)

WITH diabetes_patients AS (
  SELECT DISTINCT 
    d.patient_id,
    d.claim_id
  FROM healthverity_claims_sample_patient_dataset.hv_claims_sample.diagnosis d
  WHERE d.diagnosis_code LIKE 'E08%' 
     OR d.diagnosis_code LIKE 'E09%'
     OR d.diagnosis_code LIKE 'E10%'
     OR d.diagnosis_code LIKE 'E11%'
     OR d.diagnosis_code LIKE 'E13%'
),
claim_costs AS (
  SELECT 
    p.claim_id,
    p.patient_id,
    SUM(p.charge_amount) as total_claim_cost
  FROM healthverity_claims_sample_patient_dataset.hv_claims_sample.procedure p
  INNER JOIN diabetes_patients dp ON p.claim_id = dp.claim_id AND p.patient_id = dp.patient_id
  GROUP BY p.claim_id, p.patient_id
),
patient_age AS (
  SELECT 
    e.patient_id,
    e.birth_year,
    YEAR(CURRENT_DATE()) - e.birth_year as age,
    CASE 
      WHEN YEAR(CURRENT_DATE()) - e.birth_year < 18 THEN '0-17'
      WHEN YEAR(CURRENT_DATE()) - e.birth_year BETWEEN 18 AND 34 THEN '18-34'
      WHEN YEAR(CURRENT_DATE()) - e.birth_year BETWEEN 35 AND 49 THEN 

__Conclusion of no-tool agent version__
1. procedure.charge_amount doesn't exist at all.

In [2]:
"""
Step 1: Register Unity Catalog Functions for Metadata Querying

These UC functions will be used as tools by the LangGraph agent.
Each function queries different levels of the enriched genie docs chunks table.
"""

# Configuration
CATALOG = "yyang"
SCHEMA = "multi_agent_genie"
TABLE_NAME = f"{CATALOG}.{SCHEMA}.enriched_genie_docs_chunks"

print(f"Registering UC functions in: {CATALOG}.{SCHEMA}")
print(f"Target table: {TABLE_NAME}")
print("="*80)


Registering UC functions in: yyang.multi_agent_genie
Target table: yyang.multi_agent_genie.enriched_genie_docs_chunks

In [4]:
"""
Step 2: Create UC Functions using SQL

Register SQL UC functions that query metadata at different levels
All functions use LANGUAGE SQL for better performance and compatibility
"""

# UC Function 1: get_space_summary (SQL scalar function)
spark.sql(f"""
CREATE OR REPLACE FUNCTION {CATALOG}.{SCHEMA}.get_space_summary(
    space_ids_json STRING DEFAULT 'null' COMMENT 'JSON array of space IDs to query, or "null" to retrieve all spaces. Example: ["space_1", "space_2"] or "null"'
)
RETURNS STRING
LANGUAGE SQL
COMMENT 'Get high-level summary of Genie spaces. Returns JSON with space summaries including chunk_id, chunk_type, space_title, and content.'
RETURN
    SELECT COALESCE(
        to_json(
            map_from_entries(
                collect_list(
                    struct(
                        space_id,
                        named_struct(
                            'chunk_id', chunk_id,
                            'chunk_type', chunk_type,
                            'space_title', space_title,
                            'content', searchable_content
                        )
                    )
                )
            )
        ),
        '{{}}'
    ) as result
    FROM {TABLE_NAME}
    WHERE chunk_type = 'space_summary'
    AND (
        space_ids_json IS NULL 
        OR TRIM(LOWER(space_ids_json)) IN ('null', 'none', '')
        OR array_contains(from_json(space_ids_json, 'array<string>'), space_id)
    )
""")
print("✓ Registered: get_space_summary")

# UC Function 2: get_table_overview (SQL scalar function with grouping)
spark.sql(f"""
CREATE OR REPLACE FUNCTION {CATALOG}.{SCHEMA}.get_table_overview(
    space_ids_json STRING DEFAULT 'null' COMMENT 'JSON array of space IDs to query (required, prefer single space). Example: ["space_1"]',
    table_names_json STRING DEFAULT 'null' COMMENT 'JSON array of table names to filter, or "null" for all tables in the specified spaces. Example: ["table1", "table2"] or "null"'
)
RETURNS STRING
LANGUAGE SQL
COMMENT 'Get table-level metadata for specific Genie spaces. Returns JSON with table metadata including chunk_id, chunk_type, table_name, and content grouped by space.'
RETURN
    SELECT COALESCE(
        to_json(
            map_from_entries(
                collect_list(
                    struct(
                        space_id,
                        named_struct(
                            'space_title', space_title,
                            'tables', tables
                        )
                    )
                )
            )
        ),
        '{{}}'
    ) as result
    FROM (
        SELECT 
            space_id,
            first(space_title) as space_title,
            collect_list(
                named_struct(
                    'chunk_id', chunk_id,
                    'chunk_type', chunk_type,
                    'table_name', table_name,
                    'content', searchable_content
                )
            ) as tables
        FROM {TABLE_NAME}
        WHERE chunk_type = 'table_overview'
        AND array_contains(from_json(space_ids_json, 'array<string>'), space_id)
        AND (
            table_names_json IS NULL 
            OR TRIM(LOWER(table_names_json)) IN ('null', 'none', '')
            OR array_contains(from_json(table_names_json, 'array<string>'), table_name)
        )
        GROUP BY space_id
    )
""")
print("✓ Registered: get_table_overview")

# UC Function 3: get_column_detail (SQL scalar function with grouping)
spark.sql(f"""
CREATE OR REPLACE FUNCTION {CATALOG}.{SCHEMA}.get_column_detail(
    space_ids_json STRING DEFAULT 'null' COMMENT 'JSON array of space IDs to query (required, prefer single space). Example: ["space_1"]',
    table_names_json STRING DEFAULT 'null' COMMENT 'JSON array of table names to filter (required, prefer single table). Example: ["table1"]',
    column_names_json STRING DEFAULT 'null' COMMENT 'JSON array of column names to filter, or "null" for all columns in the specified tables. Example: ["col1", "col2"] or "null"'
)
RETURNS STRING
LANGUAGE SQL
COMMENT 'Get column-level metadata for specific Genie spaces. Returns JSON with column metadata including chunk_id, chunk_type, table_name, column_name, and content grouped by space.'
RETURN
    SELECT COALESCE(
        to_json(
            map_from_entries(
                collect_list(
                    struct(
                        space_id,
                        named_struct(
                            'space_title', space_title,
                            'columns', columns
                        )
                    )
                )
            )
        ),
        '{{}}'
    ) as result
    FROM (
        SELECT 
            space_id,
            first(space_title) as space_title,
            collect_list(
                named_struct(
                    'chunk_id', chunk_id,
                    'chunk_type', chunk_type,
                    'table_name', table_name,
                    'column_name', column_name,
                    'content', searchable_content
                )
            ) as columns
        FROM {TABLE_NAME}
        WHERE chunk_type = 'column_detail'
        AND array_contains(from_json(space_ids_json, 'array<string>'), space_id)
        AND array_contains(from_json(table_names_json, 'array<string>'), table_name)
        AND (
            column_names_json IS NULL 
            OR TRIM(LOWER(column_names_json)) IN ('null', 'none', '')
            OR array_contains(from_json(column_names_json, 'array<string>'), column_name)
        )
        GROUP BY space_id
    )
""")
print("✓ Registered: get_column_detail")

# UC Function 4: get_space_details (SQL scalar function - last resort)
spark.sql(f"""
CREATE OR REPLACE FUNCTION {CATALOG}.{SCHEMA}.get_space_details(
    space_ids_json STRING DEFAULT 'null' COMMENT 'JSON array of space IDs to query (required). Example: ["space_1", "space_2"]. WARNING: Returns large metadata - use as LAST RESORT.'
)
RETURNS STRING
LANGUAGE SQL
COMMENT 'Get complete metadata for specific Genie spaces - use as LAST RESORT (token intensive). Returns JSON with complete space metadata including chunk_id, chunk_type, space_title, and all available metadata content.'
RETURN
    SELECT COALESCE(
        to_json(
            map_from_entries(
                collect_list(
                    struct(
                        space_id,
                        named_struct(
                            'chunk_id', chunk_id,
                            'chunk_type', chunk_type,
                            'space_title', space_title,
                            'complete_metadata', searchable_content
                        )
                    )
                )
            )
        ),
        '{{}}'
    ) as result
    FROM {TABLE_NAME}
    WHERE chunk_type = 'space_details'
    AND array_contains(from_json(space_ids_json, 'array<string>'), space_id)
""")
print("✓ Registered: get_space_details")

print("\n" + "="*80)
print("✅ All 4 UC SQL functions registered successfully!")
print("="*80)


✓ Registered: get_space_summary
✓ Registered: get_table_overview
✓ Registered: get_column_detail
✓ Registered: get_space_details

✅ All 4 UC SQL functions registered successfully!

In [ ]:
"""
Step 3: Create LangGraph SQL Synthesis Agent with UC Function Toolkit

Uses Databricks LangGraph SDK with create_react_agent pattern
"""

from databricks_langchain import (
    ChatDatabricks,
    DatabricksFunctionClient,
    UCFunctionToolkit,
    set_uc_function_client,
)
from langchain.agents import create_agent
import mlflow

# Initialize Databricks Function Client
client = DatabricksFunctionClient()
set_uc_function_client(client)

# Initialize LLM
LLM_ENDPOINT_NAME = "databricks-claude-sonnet-4-5"
llm = ChatDatabricks(endpoint=LLM_ENDPOINT_NAME, temperature=0.1)

# Create UC Function Toolkit with the registered functions
uc_function_names = [
    f"{CATALOG}.{SCHEMA}.get_space_summary",
    f"{CATALOG}.{SCHEMA}.get_table_overview",
    f"{CATALOG}.{SCHEMA}.get_column_detail",
    f"{CATALOG}.{SCHEMA}.get_space_details",
]

uc_toolkit = UCFunctionToolkit(function_names=uc_function_names)
tools = uc_toolkit.tools

print(f"✓ Created UCFunctionToolkit with {len(tools)} tools:")
for tool in tools:
    print(f"  - {tool.name}")

# Create SQL Synthesis Agent (specialized for multi-agent system)
sql_synthesis_agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=(
        "You are a specialized SQL synthesis agent in a multi-agent system.\n\n"
        "ROLE: You receive execution plans from the planning agent and generate SQL queries.\n\n"
        "ASSUMPTIONS:\n"
        "- The query has been clarified and validated by upstream agents\n"
        "- The planning agent has identified relevant spaces and execution strategy\n"
        "- Your ONLY job is to synthesize the SQL query based on the plan\n\n"
        "WORKFLOW:\n"
        "1. Review the execution plan and provided metadata from planning agent\n"
        "2. If metadata is sufficient → Generate SQL immediately\n"
        "3. If insufficient, tell users you need more metadata, and reveal the thinking process including analyzing which additional part of metadata might be useful\n"
        "4. Call minimal sufficient number of UC function tools with minimal sufficient argument values to help you synthesize the SQL query; call tools in this order, you dont have to finish the order, stop when you have enough metadata:\n"
        "   a) call get_space_summary for related spaces' information including its purpose, its processing logic and how its compotent tables are related and used. Find minimal sufficent needed tables or columns, then call next tool\n"
        "   b) call get_table_overview for specific tables' schemas and relationships, if still not clear about specific columns, call next tool\n"
        "   c) call get_column_detail for wanted column details and sample values\n"
        "5. call get_space_details ONLY as last resort (token intensive!)\n\n"
        "UC FUNCTION USAGE:\n"
        "- Pass the argument values, e.g., space_ids_json STRING, table_names_json STRING, column_names_json STRING, each as a JSON array string: '[\"space_id_1\", \"space_id_2\"]' if you have specific space_ids or table_names or column_names to filter on; or pass 'null' if you need all entities under the parent level, e.g., query all tables under a space, then table_names_json should be 'null'; if you need all columns under a table, then column_names_json should be 'null'.\n"
        "- Only limit the query to spaces listed in the execution plan's relevant_space_ids. Same applied to tables and columns since they are under the space.\n"
        "- Adopt the rule of get_table_overview miminal sufficiency, i.e., if you need only two tables info in a space, call the get_table_overview with space_ids_json vale as '[\"space_id_1\"]', and table_names_json value as '[\"table_name_1\", \"table_name_2\"]', instead of passing table_names_json value as 'null' for getting all tables under a space.\n"
        "- Adopt the rule of get_column_detail miminal sufficiency, i.e., only query minimal sufficient columns details if you really need\n\n"
        "OUTPUT REQUIREMENTS:\n"
        "- Generate complete, executable SQL with:\n"
        "  * Proper JOINs based on execution plan strategy\n"
        "  * WHERE clauses for filtering\n"
        "  * Appropriate aggregations\n"
        "  * Clear column aliases\n"
        "- Return ONLY the SQL query without explanations or markdown formatting\n"
        "- If SQL cannot be generated, explain what metadata is missing"
    ),
)

print("\n" + "="*80)
print("✅ SQL Synthesis Agent created successfully!")
print("="*80)
print("\nAgent Configuration:")
print(f"  - LLM: {LLM_ENDPOINT_NAME}")
print(f"  - Tools: {len(tools)} UC functions")
print(f"  - Agent Type: LangChain Tool-Calling Agent")


ModuleNotFoundError : No module named 'databricks_langchain'

Error: [0;31m---------------------------------------------------------------------------[0m
[0;31mModuleNotFoundError[0m                       Traceback (most recent call last)
File [0;32m~/.ipykernel/49084/command--1-831337481:7[0m
[1;32m      1[0m [38;5;124;03m"""[39;00m
[1;32m      2[0m [38;5;124;03mStep 3: Create LangGraph SQL Synthesis Agent with UC Function Toolkit[39;00m
[1;32m      3[0m 
[1;32m      4[0m [38;5;124;03mUses Databricks LangGraph SDK with create_react_agent pattern[39;00m
[1;32m      5[0m [38;5;124;03m"""[39;00m
[0;32m----> 7[0m [38;5;28;01mfrom[39;00m [38;5;21;01mdatabricks_langchain[39;00m [38;5;28;01mimport[39;00m (
[1;32m      8[0m     ChatDatabricks,
[1;32m      9[0m     DatabricksFunctionClient,
[1;32m     10[0m     UCFunctionToolkit,
[1;32m     11[0m     set_uc_function_client,
[1;32m     12[0m )
[1;32m     13[0m [38;5;28;01mfrom[39;00m [38;5;21;01mlangchain[39;00m[38;5;21;01m.[39;00m[38;5;21;01magents[39;0

In [78]:
# Verify the agent is properly created
print("Verification:")
print(f"  - Agent type: {type(sql_synthesis_agent)}")
print(f"  - Agent has invoke method: {hasattr(sql_synthesis_agent, 'invoke')}")
print("\n✅ SQL Synthesis Agent is ready!")
print("✅ Configured for multi-agent system (expects Planning Agent input)")


Verification:
  - Agent type: <class 'langgraph.graph.state.CompiledStateGraph'>
  - Agent has invoke method: True

✅ SQL Synthesis Agent is ready!
✅ Configured for multi-agent system (expects Planning Agent input)

In [56]:
plan_result

{'question_clear': True,
 'sub_questions': ['Identify patients diagnosed with diabetes from diagnosis codes',
  'Calculate the cost of medical claims for diabetes patients',
  'Determine the insurance payer type for each claim',
  'Calculate patient age from birth year',
  'Group patients into age groups',
  'Calculate average claim cost by payer type and age group'],
 'requires_multiple_spaces': True,
 'relevant_space_ids': ['01f0956a4b0512e2a8aa325ffbac821b',
  '01f0956a387714969edde65458dcc22a',
  '01f0956a54af123e9cd23907e8167df9'],
 'requires_join': True,
 'join_strategy': 'fast_route',
 'execution_plan': 'Use fast_route to join across three spaces: (1) HealthVerityProcedureDiagnosis to identify diabetes patients via ICD-10 diagnosis codes and get claim costs from procedure table, (2) HealthVerityClaims to get payer type from medical_claim table, and (3) HealthVerityProviderEnrollment to get patient birth year from enrollment table to calculate age groups. Join on patient_id and c

In [ ]:
"""
Step 4: Test the SQL Synthesis Agent

Demonstrates the agent using UC functions to intelligently query metadata
"""

# Example query
example_query = "What is the average cost of medical claims for patients diagnosed with diabetes, broken down by insurance payer type and patient age group?"

# Extract space_ids from earlier plan_result or relevant_spaces
# For this example, we'll use the identified relevant spaces
if 'plan_result' in globals() and 'relevant_space_ids' in plan_result:
    space_ids_for_query = plan_result['relevant_space_ids']
else:
    # Fallback to all spaces from context
    space_ids_for_query = list(context.keys()) if 'context' in globals() else []

print("=" * 80)
print("TESTING: SQL Synthesis Agent with UC Functions")
print("=" * 80)
print(f"\nQuery: {example_query}")
print(f"\nRelevant Space IDs: {space_ids_for_query}")
print("\n" + "-" * 80)
print("Agent will:")
print("  1. Call UC functions to query metadata at appropriate levels")
print("  2. Start with space summaries, drill down as needed")
print("  3. Synthesize SQL query based on gathered metadata")
print("-" * 80 + "\n")

# Create the message for the agent
agent_message = {
    "messages": [
        {
            "role": "user",
            "content": f"""
Generate a SQL query to answer this question: {example_query}

Query Plan:
{json.dumps(plan_result, indent=2)}

Relevant Spaces metadata:
{json.dumps(relevant_spaces, indent=2)}

Use your available UC function tools to gather metadata intelligently.
Start with get_space_summary, then drill down to tables and columns as needed.
"""
        }
    ]
}

# Enable MLflow autologging for tracing
mlflow.langchain.autolog()

# Invoke the agent
print("🤖 Invoking SQL Synthesis Agent...")
print("="*80 + "\n")

result = sql_synthesis_agent.invoke(agent_message)

print("\n" + "="*80)
print("✅ AGENT EXECUTION COMPLETE")
print("="*80)
print("\nFinal Response:")
print("-"*80)
# Extract the final message content
if result and "messages" in result:
    final_message = result["messages"][-1]
    print(final_message.content)
print("-"*80)


# Extract SQL from final message
if result and "messages" in result:
    final_content = result["messages"][-1].content
    
    # Try to extract SQL from markdown code blocks if present
    if "```sql" in final_content.lower():
        # Extract content between ```sql and ```
        import re
        sql_match = re.search(r'```sql\s*(.*?)\s*```', final_content, re.IGNORECASE | re.DOTALL)
        
    elif "```" in final_content:
        # Extract any code block
        import re
        sql_match = re.search(r'```\s*(.*?)\s*```', final_content, re.DOTALL)
    
    if sql_match:
            print(sql_match.group(1).strip())

In [ ]:
"""
Step 5: Create Improved synthesize_sql_fast_route Function

Wraps the LangGraph agent into a callable function compatible with the existing workflow
"""

def synthesize_sql_fast_route_with_langgraph(
    query: str,
    plan_result: Dict[str, Any] = None,
    relevant_spaces: List[Dict[str, Any]] = None
) -> str:
    """
    Synthesize SQL query using LangGraph ReAct agent with UC function tools.
    
    Args:
        query: The user's question
        plan_result: Optional execution plan from query planning
        relevant_spaces: Optional list of relevant spaces with basic metadata
        
    Returns:
        Generated SQL query string
    """
    
    # Extract space IDs from plan_result or relevant_spaces
    space_ids = []
    if plan_result and 'relevant_space_ids' in plan_result:
        space_ids = plan_result['relevant_space_ids']
    elif relevant_spaces:
        space_ids = [s.get('space_id') for s in relevant_spaces if 'space_id' in s]
    
    # Build context string
    context_info = ""
    if plan_result:
        context_info += f"\n\nExecution Plan:\n{json.dumps(plan_result, indent=2)}"
    if relevant_spaces:
        context_info += f"\n\nRelevant Spaces:\n{json.dumps(relevant_spaces, indent=2)}"
    
    # Create agent message
    agent_message = {
        "messages": [
            {
                "role": "user",
                "content": f"""
Generate a SQL query to answer this question: {query}
{f"Relevant space IDs to investigate: {json.dumps(space_ids)}" if space_ids else ""}
{context_info}

Use your available UC function tools to gather metadata intelligently.
Start with get_space_summary, then drill down to tables and columns as needed.
Once you have sufficient metadata, provide the final SQL query.
"""
            }
        ]
    }
    
    # Invoke agent
    result = sql_synthesis_agent.invoke(agent_message)
    
    # Extract SQL from final message
    if result and "messages" in result:
        final_content = result["messages"][-1].content
        
        # Try to extract SQL from markdown code blocks if present
        if "```sql" in final_content.lower():
            # Extract content between ```sql and ```
            import re
            sql_match = re.search(r'```sql\s*(.*?)\s*```', final_content, re.IGNORECASE | re.DOTALL)
            if sql_match:
                return sql_match.group(1).strip()
        elif "```" in final_content:
            # Extract any code block
            import re
            sql_match = re.search(r'```\s*(.*?)\s*```', final_content, re.DOTALL)
            if sql_match:
                return sql_match.group(1).strip()
        
        return final_content
    
    return ""

print("✅ Created: synthesize_sql_fast_route_with_langgraph()")
print("\nThis function:")
print("  - Wraps the LangGraph SQL synthesis agent")
print("  - Compatible with existing workflow")
print("  - Uses UC functions for metadata querying")
print("  - Returns clean SQL query string")


In [ ]:
"""
Step 6: Test the Improved Function

Quick test of the wrapped function
"""

# Test with the example query
test_query = "What is the average cost of medical claims for patients diagnosed with diabetes?"

print("="*80)
print("TESTING: synthesize_sql_fast_route_with_langgraph()")
print("="*80)
print(f"\nQuery: {test_query}\n")

# Call the function
sql_result = synthesize_sql_fast_route_with_langgraph(
    query=test_query,
    plan_result=plan_result if 'plan_result' in globals() else None,
    relevant_spaces=relevant_spaces if 'relevant_spaces' in globals() else None
)

print("\n" + "="*80)
print("GENERATED SQL:")
print("="*80)
print(sql_result)
print("="*80)

print("\n✅ Function test complete!")


In [ ]:
"""
===================================================================================
MULTI-AGENT INTEGRATION FUNCTION
===================================================================================

This function wraps the SQL Synthesis Agent for use in the Super Agent workflow.
"""

def synthesize_sql_fast_route_with_langgraph(
    query: str,
    plan_result: Dict[str, Any],
    relevant_spaces: List[Dict[str, Any]] = None
) -> str:
    """
    Synthesize SQL query using SQL synthesis agent with UC function tools.
    
    **MULTI-AGENT CONTEXT**:
    This function is called by the Super Agent after:
    1. Clarification Agent validates the query
    2. Planning Agent creates execution plan
    3. This agent synthesizes SQL based on the plan
    
    Args:
        query: Original user's question (already clarified)
        plan_result: Execution plan from Planning Agent (REQUIRED):
            {
                "question_clear": bool,
                "sub_questions": list[str],
                "requires_multiple_spaces": bool,
                "relevant_space_ids": list[str],
                "requires_join": bool,
                "join_strategy": "fast_route" | "slow_route",
                "execution_plan": str
            }
        relevant_spaces: Optional metadata from vector search
        
    Returns:
        Generated SQL query string
    """
    
    # Validate - requires Planning Agent output
    if not plan_result:
        raise ValueError("plan_result required - expects Planning Agent input")
    
    # Extract plan details
    strategy = plan_result.get('join_strategy', 'unknown')
    requires_join = plan_result.get('requires_join', False)
    relevant_space_ids = plan_result.get('relevant_space_ids', [])
    
    # Build structured message
    msg = f"""Execute SQL synthesis for this query.

ORIGINAL QUERY: {query}

EXECUTION PLAN (from Planning Agent):
{json.dumps(plan_result, indent=2)}

KEY DETAILS:
- Strategy: {strategy}
- Requires Join: {requires_join}
- Relevant Spaces: {relevant_space_ids}
"""
    
    if relevant_spaces:
        msg += f"\nAVAILABLE METADATA:\n{json.dumps(relevant_spaces, indent=2)}\n"
    
    msg += "\nTASK: Generate SQL based on execution plan. Use UC tools if needed."
    
    # Invoke agent
    result = sql_synthesis_agent.invoke({"messages": [{"role": "user", "content": msg}]})
    
    # Extract SQL
    if result and "messages" in result:
        content = result["messages"][-1].content
        
        # Extract from code blocks
        if "```sql" in content.lower():
            import re
            match = re.search(r'```sql\s*(.*?)\s*```', content, re.I | re.DOTALL)
            if match:
                return match.group(1).strip()
        elif "```" in content:
            import re
            match = re.search(r'```\s*(.*?)\s*```', content, re.DOTALL)
            if match:
                return match.group(1).strip()
        
        return content
    
    return ""

print("=" * 80)
print("✅ MULTI-AGENT WRAPPER FUNCTION CREATED")
print("=" * 80)
print("\nFunction: synthesize_sql_fast_route_with_langgraph()")
print("\nWorkflow Position:")
print("  Super Agent")
print("      ↓")
print("  Clarification Agent (validates query)")
print("      ↓")  
print("  Planning Agent (creates execution plan)")
print("      ↓")
print("  SQL Synthesis Agent ← YOU ARE HERE")
print("      ↓")
print("  Returns SQL query")
print("\n" + "=" * 80)


In [ ]:
"""
===================================================================================
EXAMPLE: Testing SQL Synthesis Agent in Multi-Agent Context
===================================================================================

Simulates how the Super Agent would call this agent after Planning Agent
"""

# Example: Simulated Planning Agent output
example_query = "What is the average cost of medical claims for patients diagnosed with diabetes?"

# This would come from Planning Agent
example_plan_result = {
    "question_clear": True,
    "sub_questions": [
        "Get medical claims data",
        "Filter by diabetes diagnosis",
        "Calculate average cost"
    ],
    "requires_multiple_spaces": True,
    "relevant_space_ids": [
        "01f072dbd668159d99934dfd3b17f544__GENIE_PATIENT",
        "health_verity_claims_space_id"  # example
    ],
    "requires_join": True,
    "join_strategy": "fast_route",
    "execution_plan": "Join patient diagnosis data with claims data, filter for diabetes ICD codes, aggregate costs"
}

# Optional: metadata from vector search (would come from Planning Agent)
example_relevant_spaces = [
    {
        "space_id": "01f072dbd668159d99934dfd3b17f544__GENIE_PATIENT",
        "space_title": "GENIE Patient Data",
        "searchable_content": "Contains patient demographics, diagnoses, and clinical information"
    }
]

print("=" * 80)
print("TESTING: SQL Synthesis Agent with Planning Agent Input")
print("=" * 80)
print(f"\nOriginal Query: {example_query}")
print(f"\nPlanning Agent Output:")
print(f"  - Strategy: {example_plan_result['join_strategy']}")
print(f"  - Relevant Spaces: {len(example_plan_result['relevant_space_ids'])}")
print(f"  - Requires Join: {example_plan_result['requires_join']}")
print("\n" + "-" * 80)
print("Invoking SQL Synthesis Agent...")
print("-" * 80 + "\n")

# Enable MLflow autologging
mlflow.langchain.autolog()

# Call the function (simulating Super Agent call)
try:
    sql_result = synthesize_sql_fast_route_with_langgraph(
        query=example_query,
        plan_result=example_plan_result,
        relevant_spaces=example_relevant_spaces
    )
    
    print("\n" + "=" * 80)
    print("✅ SQL SYNTHESIS COMPLETE")
    print("=" * 80)
    print("\nGenerated SQL:")
    print("-" * 80)
    print(sql_result)
    print("-" * 80)
    
except Exception as e:
    print(f"\n❌ Error: {e}")
    print("\nNote: This is expected if UC functions or spaces don't exist yet.")
    print("The structure is correct for multi-agent integration.")


"""
===================================================================================
MULTI-AGENT SYSTEM ARCHITECTURE SUMMARY
===================================================================================

This notebook implements the SQL Synthesis Agent as part of a larger multi-agent system.

COMPLETE WORKFLOW:
══════════════════

┌─────────────────────┐
│    User Query       │
└──────────┬──────────┘
           ↓
┌─────────────────────────────────────────────────────────────────┐
│                      SUPER AGENT                                 │
│                   (LangGraph Supervisor)                         │
│  - Orchestrates all sub-agents                                  │
│  - Manages state and handoffs                                   │
│  - Returns final result to user                                 │
└──────────┬──────────────────────────────────────────────────────┘
           ↓
┌─────────────────────┐
│ 1. CLARIFICATION    │ ← Validates query clarity
│    AGENT            │   Requests clarification if needed
└──────────┬──────────┘
           ↓
┌─────────────────────┐
│ 2. PLANNING         │ ← Analyzes query
│    AGENT            │   Searches vector index
│                     │   Identifies relevant spaces
│                     │   Creates execution plan
└──────────┬──────────┘
           ↓
┌─────────────────────┐
│ 3. SQL SYNTHESIS    │ ← THIS NOTEBOOK ✨
│    AGENT            │   Receives execution plan
│    (with UC Tools)  │   Calls UC functions as needed
│                     │   Generates SQL query
└──────────┬──────────┘
           ↓
┌─────────────────────┐
│ 4. SQL EXECUTION    │ ← Executes SQL on delta tables
│    AGENT            │   Returns results
└──────────┬──────────┘
           ↓
┌─────────────────────┐
│    Final Answer     │
└─────────────────────┘


KEY COMPONENTS IN THIS NOTEBOOK:
═════════════════════════════════

1. ✅ 4 UC Functions (Registered in Unity Catalog)
   - get_space_summary
   - get_table_overview
   - get_column_detail
   - get_space_details

2. ✅ SQL Synthesis Agent (Specialized for Multi-Agent System)
   - Uses Databricks LangChain SDK (not native LangChain)
   - Created with create_agent() (updated API)
   - Has access to UC function tools
   - Focused on SQL generation only

3. ✅ Wrapper Function: synthesize_sql_fast_route_with_langgraph()
   - Receives structured input from Planning Agent
   - Validates execution plan
   - Invokes agent with proper context
   - Extracts clean SQL query

INTEGRATION WITH SUPER AGENT:
══════════════════════════════

In your agent.py file, you would integrate this as:

```python
from databricks_langchain import (
    ChatDatabricks,
    UCFunctionToolkit,
    DatabricksFunctionClient,
    set_uc_function_client
)
from langchain.agents import create_agent

# Initialize
client = DatabricksFunctionClient()
set_uc_function_client(client)
llm = ChatDatabricks(endpoint="databricks-claude-sonnet-4-5")

# Create SQL Synthesis Agent as InCodeSubAgent
sql_synthesis_subagent = InCodeSubAgent(
    tools=[
        "yyang.multi_agent_genie.get_space_summary",
        "yyang.multi_agent_genie.get_table_overview",
        "yyang.multi_agent_genie.get_column_detail",
        "yyang.multi_agent_genie.get_space_details",
    ],
    name="sql_synthesis_agent",
    description="Generates SQL queries based on execution plans using UC metadata tools"
)

# Add to supervisor
supervisor = create_langgraph_supervisor(
    llm=llm,
    in_code_agents=[
        clarification_agent,
        planning_agent,
        sql_synthesis_subagent,  # ← This agent
        sql_execution_agent
    ]
)
```

BENEFITS OF THIS ARCHITECTURE:
═══════════════════════════════

✅ Separation of Concerns - Each agent has ONE job
✅ Reusability - SQL agent can be used in different workflows  
✅ Testability - Can test SQL synthesis independently
✅ Scalability - Easy to add more agents
✅ Debugging - Clear boundaries make issues traceable
✅ Dynamic Metadata - Tools query only what's needed
✅ Governance - UC functions are versioned and governed

NEXT STEPS:
═══════════

1. Integrate into agent.py with create_langgraph_supervisor()
2. Implement Clarification and Planning agents
3. Connect to Super Agent orchestration
4. Add SQL Execution agent
5. Deploy entire system to Unity Catalog
6. Set up Databricks ResponsesAgent endpoint

See Instructions/01_overall.md for complete multi-agent requirements.
"""

print("=" * 80)
print("🎉 MULTI-AGENT SQL SYNTHESIS IMPLEMENTATION COMPLETE")
print("=" * 80)
print("\n📚 Components Created:")
print("  1. 4 UC Functions for metadata querying")
print("  2. SQL Synthesis Agent with tool-calling capability")
print("  3. Multi-agent wrapper function")
print("  4. Example test with simulated Planning Agent input")
print("\n🔗 Ready for Super Agent integration!")
print("=" * 80)


"""
SUMMARY: LangGraph SQL Synthesis Agent with UC Functions

This notebook demonstrates:

1. ✅ UC Function Registration
   - Registered 4 Python UC functions for metadata querying
   - Functions query different levels: space_summary, table_overview, column_detail, space_details
   - Registered in: {CATALOG}.{SCHEMA}

2. ✅ Databricks LangGraph SDK Integration
   - Used DatabricksFunctionClient for UC function access
   - Created UCFunctionToolkit to wrap UC functions as tools
   - Built ReAct agent with create_react_agent (NOT native LangChain)

3. ✅ Intelligent Metadata Querying
   - Agent calls UC functions dynamically based on needs
   - Starts with high-level summaries
   - Drills down to tables/columns only when needed
   - Minimizes token usage

4. ✅ MLflow Integration
   - MLflow autologging enabled for tracing
   - Agent can be logged and deployed to Unity Catalog
   - Compatible with Databricks ResponsesAgent pattern

5. ✅ Production-Ready Function
   - synthesize_sql_fast_route_with_langgraph() wraps the agent
   - Compatible with existing multi-agent system workflow
   - Returns clean SQL queries

Key Advantages over Native LangChain:
- Direct integration with Databricks infrastructure
- UC functions automatically versioned and governed
- Built-in MLflow tracking and deployment
- Compatible with Databricks ResponsesAgent and Genie integration

Next Steps:
- Integrate into Super Agent workflow (Instructions/01_overall.md)
- Deploy as part of multi-agent system
- Add to LangGraph supervisor with other agents
"""

print("=" * 80)
print("🎉 NOTEBOOK COMPLETE: LangGraph SQL Synthesis Agent")
print("=" * 80)
print("\nKey Components Created:")
print(f"  1. 4 UC Functions in {CATALOG}.{SCHEMA}")
print("  2. LangGraph ReAct Agent with UC tools")
print("  3. Production-ready synthesize_sql_fast_route_with_langgraph()")
print("\nReady for integration into the multi-agent system!")
print("=" * 80)
